# GuildLM go-dev DAPT — continued pretraining on real Go, for **$0** on Kaggle

This notebook runs **domain-adaptive continued pretraining (DAPT)** of
`Qwen2.5-Coder-7B-Instruct` on the **GuildLM mined Go core corpus**
(~45M tokens of real, permissive-licensed Go from the highest-starred GitHub
repos — built by `guild-code/go/datasets/mining/build_dapt_core.py`), using
[`anvil`](https://github.com/guildlm/anvil) on Kaggle's **free** GPU.

**Why:** SFT on the M1 plateaued at base parity — the knowledge lever is DAPT,
and the M1 is ~10× too slow for it. A free Kaggle T4/P100 is the $0 path.

**Before you Run All — three clicks:**
1. **Enable the GPU:** *Settings → Accelerator → GPU T4 x2* (or P100).
2. **Attach the corpus:** *Add Input → your `go-dapt-core` Kaggle Dataset*
   (the dataset is just the single file `go_dapt_core.jsonl`, ~172 MB —
   upload it once at kaggle.com/datasets → New Dataset).
3. *(Optional, resume)* if a previous version of this notebook trained partway,
   also *Add Input → that notebook's output* — checkpoints are picked up
   automatically and training resumes.

**Sessions & budget:** one Kaggle session is capped at ~12 h (quota 30 h/week).
A full pass over 45M tokens is likely **2–3 sessions**; checkpoints are saved
every `SAVE_STEPS` and each session resumes from the last one. `MAX_STEPS`
below caps a session so it *finishes cleanly* (checkpoint saved as notebook
output) instead of being killed at the wall.

In [ ]:
# ============================ CONFIG ======================================
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"  # 7B: the guild's chosen size

SEQ_LEN    = 1024   # 7B 4-bit fits a 16 GB T4 at seq<=1024
BATCH_SIZE = 1
GRAD_ACCUM = 16     # effective batch 16 -> 16k tokens/step with packing
LEARNING_RATE = 1.0e-4
EPOCHS     = 1      # DAPT = one pass over the corpus

SAVE_STEPS = 100    # checkpoint cadence (each ~1.6M tokens with packing)
MAX_STEPS  = 600    # cap per session (~10M tokens); -1 = run to the epoch end.
                    # Sized to finish WELL inside Kaggle's 12h batch wall
                    # (a run killed at the wall loses its outputs).
                    # ~5 sessions x 600 steps ~= the full 45M-token pass;
                    # retune from session 1's measured it/s.

OUTPUT_DIR = "/kaggle/working/go_dapt_adapter"
CORPUS_FILENAME = "go_dapt_core.jsonl"   # searched under /kaggle/input/**

ANVIL_REPO      = "https://github.com/guildlm/anvil.git"
GUILD_CODE_REPO = "https://github.com/guildlm/guild-code.git"

print("Base model:", BASE_MODEL)
print("Adapter ->", OUTPUT_DIR)

## 1. Install anvil's training stack + clone the recipe repos

Same proven setup as the go_reviewer notebook: clone anvil (package **and**
`configs/`), editable-install `[train]`, then remove Kaggle's now-mismatched
torchvision/torchaudio (text-LLM training doesn't need them; leaving them
crashes transformers with `operator torchvision::nms does not exist`).

In [ ]:
import os, sys, subprocess, pathlib

WORK = pathlib.Path("/kaggle/working")
if not WORK.is_dir():
    WORK = pathlib.Path.cwd()

here = pathlib.Path.cwd()
if (here / "src" / "train.py").is_file() and (here / "configs").is_dir():
    ANVIL_DIR = here
else:
    ANVIL_DIR = WORK / "anvil"
    if not ANVIL_DIR.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", ANVIL_REPO, str(ANVIL_DIR)], check=True)

GUILD_CODE_DIR = WORK / "guild-code"
if not GUILD_CODE_DIR.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", GUILD_CODE_REPO, str(GUILD_CODE_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ANVIL_DIR}[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=False)

if str(ANVIL_DIR) not in sys.path:
    sys.path.insert(0, str(ANVIL_DIR))

print("anvil      ->", ANVIL_DIR)
print("guild-code ->", GUILD_CODE_DIR)

## 2. Find the corpus + any previous-session checkpoints

The corpus is your attached `go-dapt-core` Kaggle Dataset. If you also attached
a previous version's notebook output, its newest `checkpoint-*` is copied into
`OUTPUT_DIR` so anvil's trainer resumes from it automatically.

In [ ]:
import glob, re, shutil

# --- corpus ---------------------------------------------------------------
hits = glob.glob(f"/kaggle/input/**/{CORPUS_FILENAME}", recursive=True)
assert hits, (
    f"{CORPUS_FILENAME} not found under /kaggle/input. Upload it as a Kaggle "
    "Dataset (New Dataset -> add the file) and attach it via Add Input."
)
CORPUS_PATH = hits[0]
print("corpus:", CORPUS_PATH, f"({os.path.getsize(CORPUS_PATH)/1e6:.0f} MB)")

# --- resume: copy the newest prior checkpoint into OUTPUT_DIR --------------
def step_of(path):
    m = re.search(r"checkpoint-(\\d+)$", path)
    return int(m.group(1)) if m else -1

prior = sorted(
    glob.glob("/kaggle/input/**/checkpoint-*", recursive=True), key=step_of
)
if prior:
    src = prior[-1]
    dst = os.path.join(OUTPUT_DIR, os.path.basename(src))
    if not os.path.isdir(dst):
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        shutil.copytree(src, dst)
    print("resuming from prior checkpoint:", dst)
else:
    print("no prior checkpoint attached — fresh DAPT run")

In [ ]:
# Peek: one corpus row (raw Go source; DAPT trains next-token on `text`).
import json
with open(CORPUS_PATH) as fh:
    row = json.loads(fh.readline())
print("keys:", sorted(row), "| source:", row.get("source"))
print(row["text"][:600])

## 3. Train — anvil `mode: pretrain` (next-token on raw Go)

Loads the committed `go_dapt.yaml` recipe, overrides it for the free-tier GPU
(7B base, fp16, seq 1024, **packing on** — packs files into full sequences so
every step is 16k real tokens), and calls anvil's real `train()`. Checkpoints
land in `OUTPUT_DIR` every `SAVE_STEPS`; if a prior checkpoint was staged above,
training **resumes** from it.

In [ ]:
from src.config import load_recipe
from src.train import train

RECIPE_PATH  = str(GUILD_CODE_DIR / "go" / "anvil" / "go_dapt.yaml")
CONFIGS_ROOT = str(ANVIL_DIR / "configs")

recipe = load_recipe(RECIPE_PATH, configs_root=CONFIGS_ROOT)

recipe.base_model.model_id            = BASE_MODEL   # recipe default is 14B
recipe.base_model.attn_implementation = None          # no FlashAttention-2 on T4/P100
recipe.dataset.path           = CORPUS_PATH
recipe.dataset.eval_path      = None
recipe.dataset.val_split      = 0.005                 # tiny held-out for loss
recipe.dataset.max_seq_length = SEQ_LEN
recipe.output_dir             = OUTPUT_DIR

recipe.sft.batch_size                  = BATCH_SIZE
recipe.sft.gradient_accumulation_steps = GRAD_ACCUM
recipe.sft.epochs                      = EPOCHS
recipe.sft.max_steps                   = MAX_STEPS
recipe.sft.learning_rate               = LEARNING_RATE
recipe.sft.save_steps                  = SAVE_STEPS
recipe.sft.packing                     = True
recipe.sft.group_by_length             = False        # packing supersedes it
recipe.sft.bf16 = False                                # T4/P100: fp16
recipe.sft.fp16 = True

print("mode      :", recipe.mode)
print("base      :", recipe.base_model.model_id)
print("seq/packing:", recipe.effective_max_seq_length, "/", recipe.sft.packing)
print("max_steps :", recipe.sft.max_steps, "| save every", recipe.sft.save_steps)

adapter_dir = train(recipe)   # the heavy step — hours
print("\nDAPT adapter saved to:", adapter_dir)

In [ ]:
# Confirm what this session produced (adapter + checkpoints become the
# notebook's output — attach it as input to the NEXT session to resume).
for root, dirs, files in os.walk(OUTPUT_DIR):
    depth = root[len(OUTPUT_DIR):].count(os.sep)
    if depth <= 1:
        print(root)
        for f in sorted(files)[:8]:
            print("   ", f)

## 4. *(Optional)* Push the adapter to the HuggingFace Hub

Add your HF **write** token under *Add-ons → Secrets* as `HF_TOKEN`, set
`PUSH = True`, and run. Free hosting; do this on the FINAL session (after the
full pass), or every session if you want off-Kaggle backups.

In [ ]:
HF_REPO_ID = "guildlm/go-dapt-7b-lora"   # <-- change if needed
PUSH = False                              # <-- set True to upload

if PUSH:
    token = os.environ.get("HF_TOKEN")
    try:
        from kaggle_secrets import UserSecretsClient
        token = token or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    from src.hub import push_adapter
    url = push_adapter(HF_REPO_ID, OUTPUT_DIR, token=token, private=True,
                       base_model=BASE_MODEL)
    print("Pushed:", url)
else:
    print("PUSH is False — skipping.")

## Next steps (back on the M1)

1. **Bridge HF → MLX:** download the adapter, `anvil-merge` it into the fp16
   base, then `mlx_lm.convert --hf-path <merged> -q` → a 4-bit MLX model the
   existing `serve_specialist.sh` / Builder stack can serve.
2. **Measure honestly:** crucible `mlx_bench` (greenfield 24) + the Builder
   project runs (usersapi/taskflow/taskapi/taskapipro) vs base 19/24.
3. **Then light SFT on top** (mixed greenfield+edit recipe) = the full
   DAPT → SFT → algorithm stack.